# 05 — Review the self-improvement loop

Run **`python improve.py`** from the project folder to advance one iteration. This notebook reviews saved results without starting another experiment.

The command evaluates the current configuration on train, diagnoses failure types, chooses an untried rule-based proposal, evaluates it, applies the declared acceptance checks, and logs the outcome. An accepted candidate becomes the reference for the next command; a rejected one leaves the reference unchanged.

It reports the original baseline as well. Neither answer spans nor negative labels enter retrieval inference. Test performance remains untouched. A finite proposal catalog makes the process reproducible and explainable; it is not an autonomous code-writing agent.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from improve import read_log, read_json, resume_state
from retrieval import BM25Retriever, load_inputs

historical = read_log(ROOT / 'experiments/log.jsonl')
history = read_log(ROOT / 'experiments/loop_log.jsonl')
assert history, 'Run python improve.py once before reviewing results.'
current, tried_ids, tried_configs, next_iteration = resume_state(
    read_json(ROOT / 'configs/bm25_baseline.json'), historical, history)
print('Distinct completed iterations:', next_iteration - 1)
print('Required minimum beyond baseline: 5')
print('Current selected approach:', current['name'])
print('Next iteration number:', next_iteration)
display(pd.DataFrame(read_json(ROOT / 'experiments/proposals.json')))

## Latest decision and baseline comparison

Acceptance compares the candidate with the current accepted configuration. Every result also retains the original baseline. The common-word experiment remains recorded separately as iteration 1; reruns of that same proposal are not new iterations.

Our primary objective is recall@5. Precision, both abstention rates, and local p95 search latency are guardrails. The baseline's zero negative abstention means the negative guardrail is initially weak; passing it does not establish safe answerability detection.

In [ ]:
latest = history[-1]
print('Iteration:', latest['iteration'])
print('Proposal:', latest['proposal']['id'])
print('Reason:', latest['proposal']['reason'])
print('Diagnosis:', latest['diagnosis_before'])
print('Accepted:', latest['decision']['accepted'])
display(pd.Series(latest['decision']['checks'], name='Passed'))
rows = []
for label, result in latest['metrics'].items():
    at5 = next(m for m in result['retrieval'] if m['k'] == 5)
    rows.append(dict(approach=label, precision_at_5=at5['precision'], recall_at_5=at5['recall'],
                     false_abstention=result['false_abstention'], correct_abstention=result['correct_abstention'],
                     p95_ms=result['latency_ms']['p95']))
display(pd.DataFrame(rows))
print('Selected configuration:')
display(pd.Series(latest['selected_config']))

## Run either saved configuration

Edit `which` to `reference_config`, `candidate_config`, or `selected_config`. Even a rejected approach can be loaded and used. The code keeps settings separate from retrieval so they can be changed during an interview.

In [ ]:
which = 'candidate_config'
_, documents, _ = load_inputs(ROOT)
model = BM25Retriever(documents, latest[which])
question = 'Under the Fox and Glu Mobile wireless content license, when does the term expire?'
hits = model.search(question, k=5)
if hits:
    display(pd.DataFrame(hits)[['score', 'file_path', 'start', 'end', 'text']])
else:
    print('No relevant passage found.')

## Explain this step

“We built a repeatable feedback loop: measure, diagnose, propose, evaluate, then accept or reject. It changes configuration using predefined rules and keeps a record of every outcome. One command advances one iteration, so we can review each decision.”

The loop supports multiple steps with `--steps N`, but we are proceeding one at a time. We still need at least five distinct iterations in total. The web app, query CLI, final report, and later LLM comparison remain separate pending work. The improvement command is not the required user-facing search CLI.